In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Function Vectors in Large Language Models - Replication

## Goal
Replicate the core experiment from "Function Vectors in Large Language Models" (ICLR 2024), demonstrating:
1. Attention heads transport compact vector representations of ICL tasks ("function vectors")
2. These vectors trigger task execution in zero-shot and shuffled-label contexts
3. Function vectors work across different prompting templates

## Methodology
- Load GPT-J 6B model and antonym task dataset
- Compute task-conditioned mean activations from ICL prompts
- Extract function vector using pre-computed universal top heads
- Evaluate on ICL, shuffled-label, zero-shot, and natural text contexts

In [2]:
# Setup and imports
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

# Repository setup
REPO_ROOT = '/net/scratch2/smallyan/function_vectors_eval'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

# Environment settings for model loading
os.environ['HF_HOME'] = '/net/scratch2/smallyan/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = '/net/scratch2/smallyan/.cache/huggingface'

# Check CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Disable gradients for inference
torch.set_grad_enabled(False)

# Reproducibility
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(42)
print("Seeds set for reproducibility")

Using device: cuda
CUDA available: True
GPU: NVIDIA A100 80GB PCIe
Seeds set for reproducibility
